**Data Info**

train.csv
* ID : 사건 샘플 ID
* first_party : 사건의 첫 번째 당사자
* second_party : 사건의 두 번째 당사자
* facts : 사건 내용
* first_party_winner : 첫 번째 당사자의 승소 여부 (0 : 패배, 1 : 승리)

test.csv
* ID : 사건 샘플 ID
* first_party : 사건의 첫 번째 당사자
* second_party : 사건의 두 번째 당사자
* facts : 사건 내용

sample_submission.csv - 제출 양식
* ID : 사건 샘플 ID
* first_party_winner : 예측한 첫 번째 당사자의 승소 여부 (0 : 패배, 1 : 승리)

# Fine-tune Model: DeBERTa-v3

In [ ]:
!pip install -U transformers
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LEN = 384
N_FOLDS = 5
SEED = 42

In [ ]:
train = pd.read_csv("./train.csv")
test = pd.read_csv("./test.csv")

In [ ]:
def build_text(row):
    # 당사자명 + facts를 하나의 시퀀스로 (BERT류는 [SEP] 대신 tokenizer가 자동 처리)
    return f"{row['first_party']} vs {row['second_party']}: {row['facts']}"

In [ ]:
train["text"] = train.apply(build_text, axis=1)
test["text"] = test.apply(build_text, axis=1)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
class JudgmentDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=MAX_LEN,
            padding=False,
        )
        item = {k: torch.tensor(v) for k, v in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

In [ ]:
X = train["text"].values
y = train["first_party_winner"].values
X_test = test["text"].values

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_pred = np.zeros(len(train))
test_logits_sum = np.zeros((len(test), 2))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    print(f"\n===== Fold {fold+1}/{N_FOLDS} =====")

    train_ds = JudgmentDataset(X[tr_idx], y[tr_idx])
    valid_ds = JudgmentDataset(X[va_idx], y[va_idx])
    test_ds = JudgmentDataset(X_test)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    )

    args = TrainingArguments(
        output_dir=f"./ckpt_fold{fold}",
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=2,   # 실질 배치 16
        num_train_epochs=5,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_steps=0.1,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",   # 대회 지표가 Accuracy이므로
        greater_is_better=True,
        bf16=True,
        fp16=False,
        logging_steps=20,
        report_to="none",
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
    )

    trainer.train()

    va_logits = trainer.predict(valid_ds).predictions
    oof_pred[va_idx] = np.argmax(va_logits, axis=1)

    test_logits = trainer.predict(test_ds).predictions
    test_logits_sum += torch.softmax(torch.tensor(test_logits), dim=1).numpy()

    acc = accuracy_score(y[va_idx], oof_pred[va_idx])
    f1 = f1_score(y[va_idx], oof_pred[va_idx], average="macro")
    print(f"Fold {fold+1} | Acc: {acc:.4f} | Macro F1: {f1:.4f}")

    del model, trainer
    torch.cuda.empty_cache()

print("\n=== Overall OOF ===")
print("Accuracy:", accuracy_score(y, oof_pred))
print("Macro F1:", f1_score(y, oof_pred, average="macro"))


===== Fold 1/5 =====


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.524825,0.972031,0.334677,0.250755
2,1.363014,0.638802,0.665323,0.399516
3,1.349664,0.646936,0.665323,0.399516
4,1.255497,0.637879,0.665323,0.399516
5,1.244400,0.637459,0.665323,0.399516


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fold 5 | Acc: 0.6646 | Macro F1: 0.3993

=== Overall OOF ===
Accuracy: 0.66545601291364
Macro F1: 0.3995638478313545


In [ ]:
test_pred = np.argmax(test_logits_sum, axis=1)
submit = pd.read_csv("./sample_submission.csv")
submit["first_party_winner"] = test_pred
submit.to_csv("./deberta_submit.csv", index=False)
print("Done")

Done
